# Lab 3: Ranking with TF-IDF, cosine similarity and BM25
**Course:** Information Retrieval (IR 2026), CADT  |  **Session 3**  |  **Time:** 3 hours  |  *Student version*

Lab 2 fixed matching (a proper inverted index) but not **ranking**: Boolean AND/OR return an unordered set of documents, all equally "correct". Today you build the two ranking models from the lecture — TF-IDF with cosine similarity, and BM25 — and see, using Lab 1's own information needs, why a ranked list is so much more useful than a set.

**By the end of this lab you can:**
1. Compute term frequency (TF), document frequency (DF) and inverse document frequency (IDF).
2. Build TF-IDF vectors and rank documents by cosine similarity to a query.
3. Implement BM25 and explain the roles of its `k1` and `b` parameters.
4. Compare TF-IDF, BM25 and Lab 2's Boolean retrieval on the same queries.
5. Explain, with your own numbers, why BM25 can rank documents differently from TF-IDF.

**How to work:** run the cells in order, replace every `# TODO`, and write your answers in the *Answer* cells. Part 6 is your Project Milestone 1 workshop — bring your team's ideas.

## Part 0. Setup and recap (about 10 min)
We reuse the same 30-document collection from Labs 1 and 2. Recall Lab 2's Boolean OR for "travel Phnom Penh Siem Reap": it returned **4 documents with no order** (`d01`, `d02`, `d12`, `d29`), even though only `d12` is truly about that trip. Keep that in mind — you will fix exactly this today.

In [ ]:
import re, math
import pandas as pd
import matplotlib.pyplot as plt
print("Setup OK")

In [ ]:
docs = {
    "d01": 'Budget guesthouses in Phnom Penh offer clean rooms near the Riverside for under twenty dollars a night.',
    "d02": 'Cheap hotel deals in Siem Reap are easy to find, especially during the rainy season.',
    "d03": 'The Cambodia Academy of Digital Technology offers degrees in software engineering, data science and cybersecurity.',
    "d04": 'Python is a popular programming language for data science, web development and automation.',
    "d05": 'A ball python is a small, docile snake that is often kept as a pet.',
    "d06": 'Java is a statically typed language widely used for enterprise applications and Android development.',
    "d07": 'Java is an island in Indonesia with more than one hundred million inhabitants.',
    "d08": 'Angkor Wat is the largest religious monument in the world and attracts millions of visitors every year.',
    "d09": 'The Mekong River flows through six countries and supports fishing and rice farming along its banks.',
    "d10": 'Rice is the main crop in Cambodia, and farmers plant it at the start of the rainy season.',
    "d11": 'Fried noodles and fish amok are popular dishes in Cambodian restaurants.',
    "d12": 'Buses run every hour from Phnom Penh to Siem Reap, and the trip takes about six hours.',
    "d13": 'Tuk-tuk drivers in the capital usually negotiate the fare before the ride starts.',
    "d14": 'Electric vehicles are becoming more common as battery prices fall.',
    "d15": 'An automobile engine converts fuel into motion using internal combustion.',
    "d16": 'Her cat sleeps all day, but at night it starts running around the house.',
    "d17": 'The library catalog lists every category of book, from fiction to computer science.',
    "d18": 'To concatenate two strings in Python, use the plus operator.',
    "d19": 'The marathon runners ran through the city streets despite the heat.',
    "d20": 'Apartments with 2 bedrooms near the university are popular with students.',
    "d21": 'Machine learning models learn patterns from data instead of following hand-written rules.',
    "d22": 'Search engines use crawlers to collect web pages and build an index for fast lookup.',
    "d23": 'A database stores structured records in tables and answers precise queries written in SQL.',
    "d24": 'The government announced new investment in digital skills and internet access for rural areas.',
    "d25": 'Coffee shops in the capital are popular places for students to study and meet friends.',
    "d26": 'Khmer script has no spaces between words, which makes text processing harder.',
    "d27": 'Airlines added more flights to the coastal city of Sihanoukville for the holiday season.',
    "d28": 'Learning to program requires regular practice, not only reading books.',
    "d29": 'The National Museum in Phnom Penh displays Khmer sculpture from the Angkorian period.',
    "d30": 'Affordable accommodation for backpackers is available in Kampot, including hostels and homestays.',
}

STOP_WORDS = set("""a an the is are was were be been being of to in on at for with and or not
this that these those it its as by from into over under about than then so if but""".split())
SUFFIXES = ["ational", "tional", "alize", "ation", "ative", "fulness", "ousness",
            "iveness", "ing", "ies", "ied", "ly", "ed", "es", "s"]

def simple_stem(word):
    for suf in SUFFIXES:
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            return word[:-len(suf)]
    return word

def preprocess(text):
    """Tokenize, remove stop words, and stem (same pipeline as Lab 2)."""
    tokens = re.findall(r"[a-z]+", text.lower())
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return [simple_stem(t) for t in tokens]

doc_tokens = {doc_id: preprocess(text) for doc_id, text in docs.items()}
doc_len = {doc_id: len(toks) for doc_id, toks in doc_tokens.items()}
N = len(docs)
AVGDL = sum(doc_len.values()) / N
print(f"{N} documents, average length {AVGDL:.1f} tokens")

## Part 1. Term frequency and document frequency (about 20 min)
**Term frequency (TF)**: how many times a term appears in ONE document.
**Document frequency (DF)**: in how many DIFFERENT documents a term appears at least once.

In [ ]:
def tf(term, doc_id): 
      """Count of `term` (already stemmed) in the given document's tokens."""
      return doc_tokens[doc_id].count(term)
  
def df(term):
      """Number of documents that contain `term` at least once."""
      return sum(1 for doc_id in doc_tokens if term in doc_tokens[doc_id])


In [ ]:
for term in ["cambodia", "offer", "python", "hotel", "phnom", "cheap", "siem"]:
    print(f"{term:>10}: df = {df(term)}")
    
for term in ["cambodia", "offer", "python", "hotel", "phnom", "cheap", "siem"]:
    counts = {doc_id: tf(term, doc_id) for doc_id in doc_tokens if tf(term, doc_id) > 0}
    print(f"{term:>10}: tf = {counts}")

**Answer 1.1** Which of these terms is the most common across the collection, and which is the rarest? If a query contained both a common and a rare term, which one should matter more for ranking, and why?

*Your answer:* Of these terms, the most common across the collection is `python` (df=3), and the rarest are `hotel` and `cheap` (each appears in only one document). If a query contained both a common and a rare term, the rare term should matter more for ranking. Because a rare term appears in very few documents and is highly selective — it is much stronger evidence that a document is relevant to the query.

## Part 2. IDF and TF-IDF vectors (about 25 min)
A term that appears in every document (high DF) tells us almost nothing about relevance. **Inverse document frequency (IDF)** down-weights common terms:

$$\text{idf}(t) = \log\left(\frac{N}{\text{df}(t)}\right)$$

**TF-IDF** combines both: `tf(t, d) * idf(t)`. A term scores high when it is frequent IN this document but rare ACROSS the collection.

In [ ]:
def idf(term):
    """Calculate inverse document frequency."""
    d = df(term)
    if d == 0:
        return 0.0
    return math.log(N / d)


def tfidf_vector(doc_id):
    """Build a TF-IDF vector for a document."""
    vector = {}
    for term in set(doc_tokens[doc_id]):
        vector[term] = tf(term, doc_id) * idf(term)
    return vector

In [ ]:
for term in ["cambodia", "offer", "python", "hotel", "phnom", "cheap", "siem"]:
    print(f"{term:>10}: idf = {idf(term):.3f}")

print()
print("TF-IDF vector for d02:", {k: round(v, 3) for k, v in tfidf_vector("d02").items()})

**Answer 2.1** `offer` and `cambodia` have the same document frequency. Do they get the same IDF? Should they? What does this tell you about what IDF does and does not capture?

*Your answer:* Yes, both `offer` and `cambodia` get the same IDF (≈2.708) because they both appear in exactly 2 documents. IDF depends only on how many documents contain a term, not on the term's meaning or importance. This tells us that IDF captures statistical rarity across the collection but does NOT capture semantic meaning — two completely different words with the same document frequency are treated identically.

## Part 3. Cosine similarity and ranked retrieval (about 30 min)
Treat every document and every query as a vector of TF-IDF weights. **Cosine similarity** measures the angle between two vectors: 1.0 means identical direction (very similar), 0.0 means no shared terms.

$$\cos(\vec q, \vec d) = \frac{\vec q \cdot \vec d}{\|\vec q\| \, \|\vec d\|}$$

In [ ]:
def cosine_similarity(v1, v2):
    """Cosine similarity between two {term: weight} vectors. Return 0.0 if either norm is 0."""
    # dot product over shared terms only
    dot = sum(v1[t] * v2[t] for t in v1 if t in v2)
    
    norm1 = math.sqrt(sum(val**2 for val in v1.values()))
    norm2 = math.sqrt(sum(val**2 for val in v2.values()))
    
    if norm1 == 0 or norm2 == 0:
        return 0.0
    
    return dot / (norm1 * norm2)

In [ ]:
def query_vector(query_text):
    """Build a TF-IDF vector for the query itself, using the SAME idf() as the collection."""
    terms = preprocess(query_text)
    tf_q = {}
    for t in terms:
        tf_q[t] = tf_q.get(t, 0) + 1
    return {t: tf_q[t] * idf(t) for t in tf_q}

def rank_tfidf(query_text, top_k=10):
    """Return [(doc_id, score), ...] sorted by cosine similarity, highest first, score > 0 only."""
    q_vec = query_vector(query_text)
    
    scores = []
    for doc_id in doc_tokens:
        d_vec = tfidf_vector(doc_id)
        score = cosine_similarity(q_vec, d_vec)
        if score > 0:
            scores.append((doc_id, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

In [ ]:
for q in ["cheap hotel", "python", "travel Phnom Penh Siem Reap", "car"]:
    print(f"\n{q!r} ->")
    for doc_id, score in rank_tfidf(q):
        print(f"   {doc_id}: {score:.3f}  {docs[doc_id][:60]}...")

**Answer 3.1** Compare the ranked list for "travel Phnom Penh Siem Reap" with Lab 2's Boolean OR result (`d01`, `d02`, `d12`, `d29`, unordered). Is the truly relevant document (`d12`) now easier to find? Why?

*Your answer:* Yes, `d12` is now much easier to find — it ranks #1 with a cosine score of 0.440, well above `d02` (0.280), `d29` (0.226), and `d01` (0.198). Boolean OR returned all four documents as equally valid with no ordering, so a user had to read all of them. TF-IDF cosine similarity rewards `d12` because it contains both "phnom penh" AND "siem reap" (two of the query's rare terms), while `d01` only has "phnom penh" and `d02` only has "siem reap". The more query terms a document matches, and the rarer those terms are, the higher it scores.

**Answer 3.2** "car" still returns nothing. Explain why, in one sentence, using a term from the lecture (Session 1–2).

*Your answer:* The query "car" returns no results because of the **vocabulary mismatch** problem — the word "car" does not appear in any document (documents use "automobile", "vehicle", "engine" instead), so there are no shared terms and the cosine similarity is 0 for every document.

## Part 4. BM25: term saturation and length normalization (about 30 min)
TF-IDF grows the score **linearly** with term frequency: 10 occurrences count 10 times as much as 1. In practice, that overrates repetition. BM25 fixes this with **saturation** (extra occurrences matter less and less) and adds **document length normalization** (a term match in a short document is stronger evidence than the same match in a long one).

First, see saturation on its own, without building the full formula yet.

In [ ]:
demo_docs = {
    "p1": "coffee is nice",
    "p2": "coffee coffee coffee is nice and coffee is popular",
    "p3": ("coffee " * 10) + "is popular",
}
demo_tf = {k: v.split().count("coffee") for k, v in demo_docs.items()}

def saturate(f, k1=1.5):
    """BM25's saturation curve for a single term, ignoring length normalization (b=0)."""
    return (f * (k1 + 1)) / (f + k1)

for k, f in demo_tf.items():
    print(f"{k}: raw tf = {f:>2}  |  raw tf scaled to /10 = {f/10:.2f}  |  BM25 saturation = {saturate(f):.2f}")

**Answer 4.1** `p3` has 10 times the raw term frequency of `p1`, but its saturation score is nowhere near 10 times higher. What real-world problem does this solve? (Hint: think about a spammy document that repeats a keyword 100 times.)

*Your answer:* This solves the keyword spamming problem. Without saturation, a web page that repeats a keyword 100 times would score 100× higher than a page that mentions it once meaningfully. BM25's saturation curve means that after the first few occurrences, each additional occurrence contributes very little to the score. This prevents low-quality documents that simply stuff keywords from ranking above genuinely relevant documents that use a term naturally.

Now implement the full BM25 score, including length normalization:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{idf}_{\text{BM25}}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \cdot \left(1 - b + b \cdot \dfrac{|d|}{\text{avgdl}}\right)}$$

where `idf_BM25(t) = log((N - df(t) + 0.5) / (df(t) + 0.5) + 1)` (a variant of IDF used specifically in BM25) and `|d|` is the document's length in tokens.

In [ ]:
def idf_bm25(term):
    """BM25's IDF variant (always defined, even for a term that is in every document)."""
    d = df(term)
    return math.log((N - d + 0.5) / (d + 0.5) + 1)

def bm25_score(query_text, doc_id, k1=1.5, b=0.75):
    """BM25 score of one document for a query."""
    terms = set(preprocess(query_text))
    score = 0.0
    for term in terms:
        f = tf(term, doc_id)
        if f > 0:
            score += idf_bm25(term) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * doc_len[doc_id] / AVGDL))
    return score

def rank_bm25(query_text, top_k=10, k1=1.5, b=0.75):
    """Return [(doc_id, score), ...] sorted by BM25 score, highest first, score > 0 only."""
    scores = []
    for doc_id in doc_tokens:
        score = bm25_score(query_text, doc_id, k1=k1, b=b)
        if score > 0:
            scores.append((doc_id, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

In [ ]:
for q in ["cheap hotel", "python", "travel Phnom Penh Siem Reap"]:
    print(f"\n{q!r} -> BM25")
    for doc_id, score in rank_bm25(q):
        print(f"   {doc_id}: {score:.3f}  {docs[doc_id][:60]}...")

## Part 5. TF-IDF vs. BM25: same query, different order? (about 20 min)

In [ ]:
print("Document lengths (tokens):", {d: doc_len[d] for d in ["d04", "d05", "d18"]})
print(f"Average document length: {AVGDL:.1f}\n")

print("Query: 'python'")
print("  TF-IDF ranking:", rank_tfidf("python"))
print("  BM25 ranking:  ", rank_bm25("python"))

**Answer 5.1** TF-IDF and BM25 do NOT agree on the top result for "python". Using the document lengths printed above, explain why. Which model's reasoning do you find more convincing for a one-word query, and why?

*Your answer:* TF-IDF ranks `d04` first (score 0.285) because all three documents contain "python" exactly once, so their raw TF-IDF scores differ only by the length of their vectors — `d04` has 9 tokens and a longer vector that somewhat dilutes its norm, but it still comes out slightly ahead. BM25 flips the order and ranks `d18` first (score 2.471) because `d18` is the shortest document (7 tokens, below the average of 9.5). BM25's length normalization with `b=0.75` penalizes longer documents, so the same single occurrence of "python" in a shorter document is treated as stronger evidence of relevance. BM25's reasoning is more convincing here: finding "python" in a 7-word sentence (d18: "To concatenate two strings in Python, use the plus operator") suggests the document is specifically about Python, whereas in a longer document the term could be incidental.

In [ ]:
# Try at least 3 different (k1, b) combinations on the query "python" and observe the ranking.
# Suggested values: b in {0, 0.75, 1.0}; k1 in {0.5, 1.5, 3.0}
for b in [0, 0.75, 1.0]:
    print(f"b={b}:", rank_bm25("python", k1=1.5, b=b))

**Answer 5.2** What happens to the ranking when `b = 0`? Using the formula, explain why. What does `b` control, in your own words?

*Your answer:* When `b = 0`, all three documents (`d04`, `d05`, `d18`) receive exactly the same BM25 score (2.181) and the ranking becomes a tie. Looking at the formula: when `b = 0`, the denominator simplifies to `f + k1 * (1 - 0 + 0) = f + k1`, which is the same for all three documents since they each have `f=1` for "python". The document length term `b * |d| / avgdl` disappears completely. In plain terms, `b` controls **how much document length affects the score**. When `b = 0`, length is completely ignored. When `b = 1.0`, a document twice the average length is penalized heavily. In practice `b = 0.75` is a balance: longer documents are penalized, but not as severely as full normalization.

**Answer 5.3** Run the six Lab 1 information needs (N1–N6) through both `rank_tfidf` and `rank_bm25` (loop over the `needs` dictionary below, like you did in Lab 2 Part 6). Pick ONE need where the top-ranked result is NOT the document you judged relevant in Lab 1. What is still going wrong — is it a ranking problem, or the same vocabulary-mismatch problem from Sessions 1–2?

*Your answer:* For **N4** ("Information about cars", query: "car"), both `rank_tfidf` and `rank_bm25` return **no results at all**. The relevant documents (`d14`, `d15`) use words like "automobile", "vehicle", "engine", "electric" — but the query uses the word "car", which never appears in any document. This is the same **vocabulary mismatch** problem from Sessions 1–2: the ranking models are working correctly but they cannot rank documents that share no terms with the query. This will be addressed in future sessions with query expansion, synonym handling, or semantic search techniques (e.g., dense retrieval with word embeddings).

In [ ]:
needs = {
    "N1": {"need": 'Find a cheap place to stay in Cambodia', "queries": ['cheap hotel', 'budget accommodation']},
    "N2": {"need": 'Learn about the Python programming language', "queries": ['python']},
    "N3": {"need": 'Travel from Phnom Penh to Siem Reap', "queries": ['travel Phnom Penh Siem Reap']},
    "N4": {"need": 'Information about cars', "queries": ['car']},
    "N5": {"need": 'Documents about cats', "queries": ['cat']},
    "N6": {"need": 'Text about running', "queries": ['running']},
}

for nid, info in needs.items():
    for q in info["queries"]:
        print(nid, q)
        print("  tfidf:", rank_tfidf(q, top_k=3))
        print("  bm25 :", rank_bm25(q, top_k=3))

## Part 6. Project Milestone 1 workshop (about 25 min, in your team)
Your project proposal is due this week. Work with your team to fill in the template below (copy it into a shared document, not this notebook), then swap with another team for 5 minutes of feedback before you submit.

**Project Milestone 1 — Proposal template**

1. **Problem**: what information need will your search engine serve? Who is the user?
2. **Corpus**: what will you collect (topic, language, approximate source), and roughly how many documents or pages do you expect?
3. **Team roles**: who owns crawling/scraping, indexing, ranking/evaluation, and the write-up? (Sessions 5–6 will need your crawler ready.)
4. **Risks**: what is the biggest uncertainty right now (access to the site, Khmer text processing, corpus size)?

*Submit your team's proposal as instructed by the end of the session.*

## Part 7. Reflection (about 5 min in class; finish as homework)
Write 4–6 sentences: *What can a ranked list do that Lab 2's Boolean AND/OR cannot? Use one concrete example from Part 3 or Part 5. Then name one thing that is STILL not solved by ranking (Part 5, Answer 5.3) and which future session will address it.*

*Your reflection:* A ranked list orders results by estimated relevance so that the most useful document appears first, whereas Boolean AND/OR can only return an unordered set of all documents that match or do not match. This is a critical improvement for users: for the query "travel Phnom Penh Siem Reap", Boolean OR returned `d01`, `d02`, `d12`, and `d29` with no indication of which is best, but TF-IDF cosine similarity placed `d12` — the document actually about that bus route — at rank 1 with score 0.440, well ahead of the others. BM25 further reinforced this by giving `d12` an even larger lead (8.388 vs 4.693 for second place) because it matches the most query terms. However, ranking cannot solve vocabulary mismatch: queries like "car" still return nothing because the collection uses "automobile" and "vehicle" instead, and no ranking model can score a document with zero shared terms. This limitation will be addressed in future sessions through techniques like query expansion or semantic/dense retrieval using word embeddings.

---
### Submission checklist
- [x] All `# TODO` cells completed and the notebook runs from top to bottom without errors
- [x] All *Answer* cells written in your own words
- [x] Part 5 comparison across N1–N6 completed
- [ ] Project Milestone 1 proposal submitted
- [ ] Notebook saved as `Lab3_<StudentID>.ipynb` and submitted as instructed